# watsonx.data — Customer Table: Time Travel & Snapshot Management Demo

## What this notebook does

This notebook walks through a complete, end-to-end scenario using an **Apache Iceberg** table (`tcustomers_tt`) in **IBM watsonx.data**. It covers three main areas:

---

### 1. Table creation and incremental data loading
- Creates an Iceberg v2 table (`tcustomers_tt`) with a realistic customer schema (name, email, location, purchase history, status).
- Writes data in **three separate INSERT batches** (5 + 3 + 4 customers). Each INSERT produces a new **snapshot** in the Iceberg metadata — this is the foundation for time travel.

---

### 2. Snapshots
Every write operation on an Iceberg table is recorded as an immutable **snapshot**. A snapshot captures:
- The exact set of data files that make up the table at that point in time.
- Metadata such as `snapshot_id`, `committed_at` timestamp, operation type, and record counts.

This notebook demonstrates how to:
- **Inspect the snapshot history** via the `<table>.snapshots` metadata table to see all versions created by the inserts.
- **Configure snapshot retention policies** using `TBLPROPERTIES`:
  - `history.expire.max-snapshot-age-ms` — automatically expire snapshots older than 7 days.
  - `history.expire.min-snapshots-to-keep` — always retain at least 5 snapshots regardless of age.
- **Manually expire old snapshots** using the Iceberg stored procedure `CALL <catalog>.system.expire_snapshots(...)`, which physically removes snapshot metadata (and eligible data files) older than a specified timestamp while keeping at least N recent snapshots.

---

### 3. Time Travel
Because Iceberg retains snapshot history, you can **query the table as it existed at any past point in time**. This notebook shows:
- **`VERSION AS OF <snapshot_id>`** — query the table at a specific snapshot ID to see exactly which rows existed after, for example, only Batch 1 had been loaded.
- Comparing the historical view (only 5 customers after Batch 1) against the current state (12 customers after all three batches).
- Viewing `metadata_log_entries` to understand the Iceberg metadata file history.

---

### Notebook flow at a glance

| Step | What happens |
|------|--------------|
| 1 | Initialise Spark session with watsonx.data HMS credentials |
| 2 | Verify metastore connectivity |
| 3 | Discover available catalogs |
| 4–5 | Configure catalog/schema and create the `tcustomers_tt` Iceberg table |
| 6–9 | Insert customers in 3 batches → 3 snapshots are created |
| 10 | Inspect the snapshot history |
| 11–12 | Set retention policies (max age 7 days, min 5 snapshots) |
| 13 | Manually expire snapshots older than a given timestamp |
| 14 | **Time travel** — query the table `VERSION AS OF` the first snapshot |
| 15 | Inspect metadata log entries |
| 16 | Summary |

---

**Prerequisites:**
- IBM watsonx.data instance with Hive Metastore (HMS) configured
- Valid watsonx.data username and API key
- Watson Studio Spark runtime or a PySpark environment with Iceberg extensions available

## 1. Initialize Spark Session with watsonx.data Credentials

This cell captures the existing Spark configuration, stops the session, adds watsonx.data credentials, and recreates the session.

In [ ]:
import base64
import getpass
from pyspark.sql import SparkSession

# 1) Capture current Spark conf (created by Watson Studio runtime)
conf = spark.sparkContext.getConf()

# 2) Stop existing Spark session
spark.stop()

# 3) Prompt for credentials
wxd_username = getpass.getpass("Please enter your watsonx.data username (HMS access): ").strip()

# In many watsonx.data setups, HMS expects username in this format:
wxd_hms_username = "ibmlhapikey_" + wxd_username

# API key used for metastore access (HMS)
wxd_hms_password = getpass.getpass("Please enter your watsonx.data API key (HMS access): ").strip()

# watsonx.data API key header format used by the Spark extension
string_to_encode = f"{wxd_username}:{wxd_hms_password}"
wxd_encoded_apikey = "ZenApiKey " + base64.b64encode(string_to_encode.encode("utf-8")).decode("utf-8")

# 4) Inject the required properties into the existing conf
conf.setAll([
    ("spark.hive.metastore.client.plain.username", wxd_hms_username),
    ("spark.hive.metastore.client.plain.password", wxd_hms_password),
    ("spark.hadoop.hive.wxd.user.name", wxd_username),
    ("spark.hadoop.wxd.apikey", wxd_encoded_apikey),

    # REQUIRED for Iceberg procedures like CALL ... expire_snapshots
    ("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
])

# 5) Recreate Spark session using the modified conf
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

print("✓ Credentials configured successfully")
print("spark.sql.extensions =", spark.conf.get("spark.sql.extensions", "NOT_SET"))
print("Spark created:", spark)
print("spark.sql.catalogImplementation =", spark.conf.get("spark.sql.catalogImplementation", "n/a"))
print("spark.hive.metastore.uris =", spark.conf.get("spark.hive.metastore.uris", "n/a"))

## 2. Verify Metastore Connectivity

In [ ]:
try:
    # List all databases in the metastore
    dbs = spark.catalog.listDatabases()
    
    print(f"✅ Metastore reachable. Databases found: {len(dbs)}")
    print("\nAvailable databases:")
    
    for d in dbs[:50]:  # Limit to first 50 databases
        print(f"  - {d.name}")
        
except Exception as e:
    print("❌ Metastore NOT reachable")
    print(f"Exception type: {type(e).__name__}")
    print(f"Error: {str(e)}")
    
    # Try to expose more detail if Spark attached it
    if hasattr(e, "desc"):
        print("\n--- e.desc ---")
        print(e.desc)
    if hasattr(e, "stackTrace"):
        print("\n--- e.stackTrace (first 2500 chars) ---")
        print(str(e.stackTrace)[:2500])
    
    raise

## 3. Discover Available Catalogs

In [ ]:
import re

conf_items = dict(spark.sparkContext.getConf().getAll())
catalog_pattern = re.compile(r"^spark\.sql\.catalog\.([^.]+)(?:\..+)?$")

catalogs = sorted({m.group(1) for k in conf_items.keys() if (m := catalog_pattern.match(k))} | {"spark_catalog"})

print("Catalogs discovered from Spark config:")
for c in catalogs:
    print(f"  - {c}")

## 4. Define Schema and Table Configuration

**Important:** Adjust the catalog name based on your environment. Common options:
- `iceberg_data` (for iceberg-data catalog)
- `sbucket01` (for OCS bucket catalog)
- Use backticks if catalog name contains hyphens: `` `iceberg-data` ``

In [ ]:
# Define catalog and schema details - ADJUST THESE TO YOUR ENVIRONMENT
catalog_name = "iceberg_data"  # or "sbucket01" or "`iceberg-data`"
schema_name = "customer_demo"
table_name = "tcustomers_tt"

# Full table reference
full_table = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Table configuration:")
print(f"  Catalog: {catalog_name}")
print(f"  Schema: {schema_name}")
print(f"  Table: {table_name}")
print(f"  Full reference: {full_table}")

## 5. Set Current Catalog and Create Schema

Set the current catalog programmatically and create the schema.

In [ ]:
# Set the current catalog programmatically
try:
    spark.catalog.setCurrentCatalog(catalog_name)
    print(f"✓ Current catalog set to: {catalog_name}")
    
    # Now create schema with just the schema name
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {schema_name}")
    print(f"✓ Schema ensured: {catalog_name}.{schema_name}")
    
except Exception as e:
    print(f"⚠ Could not set catalog to {catalog_name}: {str(e)}")
    print(f"\nPlease verify:")
    print(f"  1. The catalog name '{catalog_name}' exists in your environment")
    print(f"  2. Check available catalogs in cell #3")
    print(f"  3. Update the catalog_name variable in cell #4 if needed")
    print(f"\nCommon catalog names: iceberg_data, sbucket01, hive_data")
    raise

## 6. Create Customer Table

Create the `tcustomers_tt` table with customer information fields and Merge-on-Read configuration.

In [ ]:
print(f"Creating table: {full_table}")

# Create customer table with comprehensive schema
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {full_table} (
        customer_id INT COMMENT 'Unique customer identifier',
        first_name STRING COMMENT 'Customer first name',
        last_name STRING COMMENT 'Customer last name',
        email STRING COMMENT 'Customer email address',
        phone STRING COMMENT 'Customer phone number',
        city STRING COMMENT 'Customer city',
        state STRING COMMENT 'Customer state',
        country STRING COMMENT 'Customer country',
        registration_date DATE COMMENT 'Customer registration date',
        last_purchase_date DATE COMMENT 'Last purchase date',
        total_purchases DOUBLE COMMENT 'Total purchase amount',
        status STRING COMMENT 'Customer status (active, inactive, suspended)',
        created_ts TIMESTAMP COMMENT 'Record creation timestamp',
        updated_ts TIMESTAMP COMMENT 'Record update timestamp'
    )
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
""")

print(f"✓ Table created: {full_table}")

# Verify table exists and show schema
print("\nTable schema:")
spark.sql(f"DESCRIBE {full_table}").show(truncate=False)

## 7. Insert Initial Customer Data - Batch 1

In [ ]:
print(f"Inserting initial customer data (Batch 1)...")

spark.sql(f"""
    INSERT INTO {full_table} VALUES
        (1, 'John', 'Smith', 'john.smith@email.com', '+1-555-0101', 'New York', 'NY', 'USA', 
         DATE '2024-01-15', DATE '2026-06-10', 1250.50, 'active', 
         TIMESTAMP '2024-01-15 10:30:00', TIMESTAMP '2026-06-10 14:20:00'),
        (2, 'Emma', 'Johnson', 'emma.j@email.com', '+1-555-0102', 'Los Angeles', 'CA', 'USA', 
         DATE '2024-02-20', DATE '2026-06-15', 2340.75, 'active', 
         TIMESTAMP '2024-02-20 09:15:00', TIMESTAMP '2026-06-15 11:45:00'),
        (3, 'Michael', 'Brown', 'mbrown@email.com', '+1-555-0103', 'Chicago', 'IL', 'USA', 
         DATE '2024-03-10', DATE '2026-05-28', 890.25, 'active', 
         TIMESTAMP '2024-03-10 16:00:00', TIMESTAMP '2026-05-28 10:30:00'),
        (4, 'Sophia', 'Davis', 'sophia.davis@email.com', '+1-555-0104', 'Houston', 'TX', 'USA', 
         DATE '2024-04-05', DATE '2026-06-18', 3120.00, 'active', 
         TIMESTAMP '2024-04-05 13:20:00', TIMESTAMP '2026-06-18 15:10:00'),
        (5, 'James', 'Wilson', 'jwilson@email.com', '+1-555-0105', 'Phoenix', 'AZ', 'USA', 
         DATE '2024-05-12', DATE '2026-06-12', 1567.80, 'active', 
         TIMESTAMP '2024-05-12 11:45:00', TIMESTAMP '2026-06-12 09:25:00')
""")

print("✓ Batch 1 inserted successfully (5 customers)")

# Display current data
print(f"\nCurrent data in {full_table}:")
spark.sql(f"SELECT customer_id, first_name, last_name, email, city, status, total_purchases FROM {full_table} ORDER BY customer_id").show(truncate=False)

## 8. Insert Additional Customer Data - Batch 2

In [ ]:
print(f"Inserting additional customer data (Batch 2)...")

spark.sql(f"""
    INSERT INTO {full_table} VALUES
        (6, 'Olivia', 'Martinez', 'olivia.m@email.com', '+1-555-0106', 'Philadelphia', 'PA', 'USA', 
         DATE '2024-06-08', DATE '2026-06-20', 2890.45, 'active', 
         TIMESTAMP '2024-06-08 14:30:00', TIMESTAMP '2026-06-20 16:40:00'),
        (7, 'William', 'Garcia', 'wgarcia@email.com', '+1-555-0107', 'San Antonio', 'TX', 'USA', 
         DATE '2024-07-15', DATE '2026-06-05', 1234.90, 'active', 
         TIMESTAMP '2024-07-15 10:20:00', TIMESTAMP '2026-06-05 12:15:00'),
        (8, 'Ava', 'Rodriguez', 'ava.r@email.com', '+1-555-0108', 'San Diego', 'CA', 'USA', 
         DATE '2024-08-22', DATE '2026-06-22', 4567.30, 'active', 
         TIMESTAMP '2024-08-22 09:45:00', TIMESTAMP '2026-06-22 14:55:00')
""")

print("✓ Batch 2 inserted successfully (3 customers)")

# Display updated data
print(f"\nUpdated data in {full_table}:")
spark.sql(f"SELECT customer_id, first_name, last_name, city, state, total_purchases FROM {full_table} ORDER BY customer_id").show(truncate=False)

## 9. Insert More Customer Data - Batch 3

In [ ]:
print(f"Inserting more customer data (Batch 3)...")

spark.sql(f"""
    INSERT INTO {full_table} VALUES
        (9, 'Liam', 'Hernandez', 'liam.h@email.com', '+1-555-0109', 'Dallas', 'TX', 'USA', 
         DATE '2024-09-10', DATE '2026-06-08', 987.65, 'active', 
         TIMESTAMP '2024-09-10 15:10:00', TIMESTAMP '2026-06-08 11:20:00'),
        (10, 'Isabella', 'Lopez', 'isabella.l@email.com', '+1-555-0110', 'San Jose', 'CA', 'USA', 
         DATE '2024-10-18', DATE '2026-06-19', 3456.20, 'active', 
         TIMESTAMP '2024-10-18 12:30:00', TIMESTAMP '2026-06-19 13:45:00'),
        (11, 'Noah', 'Gonzalez', 'noah.g@email.com', '+1-555-0111', 'Austin', 'TX', 'USA', 
         DATE '2024-11-25', DATE '2026-06-14', 2109.85, 'active', 
         TIMESTAMP '2024-11-25 08:50:00', TIMESTAMP '2026-06-14 10:05:00'),
        (12, 'Mia', 'Perez', 'mia.perez@email.com', '+1-555-0112', 'Jacksonville', 'FL', 'USA', 
         DATE '2024-12-05', DATE '2026-06-16', 1678.40, 'inactive', 
         TIMESTAMP '2024-12-05 14:15:00', TIMESTAMP '2026-06-16 09:30:00')
""")

print("✓ Batch 3 inserted successfully (4 customers)")

# Display final data count
count = spark.sql(f"SELECT COUNT(*) as total_customers FROM {full_table}").collect()[0]['total_customers']
print(f"\nTotal customers in table: {count}")

# Show all data
print(f"\nAll customer data:")
spark.sql(f"SELECT * FROM {full_table} ORDER BY customer_id").show(truncate=False)

## 10. View Table Snapshots

Check the snapshot history to see all versions created by our inserts.

In [ ]:
print(f"Snapshot history for {full_table}:")
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.{table_name}.snapshots ORDER BY committed_at").show(truncate=False)

print(f"\nSnapshot summary:")
spark.sql(f"""
    SELECT 
        snapshot_id,
        committed_at,
        operation,
        summary['added-records'] as added_records,
        summary['total-records'] as total_records
    FROM {catalog_name}.{schema_name}.{table_name}.snapshots 
    ORDER BY committed_at
""").show(truncate=False)

## 11. Configure Retention Policy - Snapshot Age

Set retention to 7 days of snapshot history (604800000 milliseconds = 7 days).

In [ ]:
print(f"Setting retention policy for {full_table}...")

# Set retention to 7 days of snapshot history
spark.sql(f"""
    ALTER TABLE {full_table} SET TBLPROPERTIES (
        'history.expire.max-snapshot-age-ms' = '604800000'
    )
""")

print("✓ Retention policy set: 7 days of snapshot history")

# Verify the property was set
print("\nTable properties (retention related):")
spark.sql(f"""
    SHOW TBLPROPERTIES {full_table}
""").filter("key LIKE '%history%' OR key LIKE '%expire%'").show(truncate=False)

## 12. Configure Additional Retention Properties

Set additional retention policies for better snapshot management.

In [ ]:
print(f"Setting additional retention properties...")

# Set minimum number of snapshots to retain and max snapshot age
spark.sql(f"""
    ALTER TABLE {full_table} SET TBLPROPERTIES (
        'history.expire.min-snapshots-to-keep' = '5',
        'history.expire.max-snapshot-age-ms' = '604800000'
    )
""")

print("✓ Additional retention properties set:")
print("  - Minimum snapshots to keep: 5")
print("  - Maximum snapshot age: 7 days")

# Show all table properties
print("\nAll table properties:")
spark.sql(f"SHOW TBLPROPERTIES {full_table}").show(truncate=False)

## 13. Manually Expire Old Snapshots

Use the Iceberg procedure to manually expire snapshots older than a specific timestamp.

In [ ]:
print(f"Manually expiring old snapshots...")

# Note: Adjust the timestamp and retain_last parameters as needed
# This example expires snapshots older than November 1, 2024, but retains at least 100 snapshots

try:
    spark.sql(f"""
        CALL {catalog_name}.system.expire_snapshots(
            table => '{schema_name}.{table_name}',
            older_than => TIMESTAMP '2024-11-01 00:00:00',
            retain_last => 100
        )
    """)
    
    print("✓ Snapshot expiration procedure executed successfully")
    print("  - Expired snapshots older than: 2024-11-01 00:00:00")
    print("  - Retained at least: 100 most recent snapshots")
    
except Exception as e:
    print(f"⚠ Note: Snapshot expiration may not have removed any snapshots")
    print(f"  Reason: {str(e)}")
    print("  This is normal if all snapshots are newer than the specified timestamp")

# Show remaining snapshots
print("\nRemaining snapshots after expiration:")
spark.sql(f"SELECT snapshot_id, committed_at, operation FROM {catalog_name}.{schema_name}.{table_name}.snapshots ORDER BY committed_at").show(truncate=False)

## 14. Demonstrate Time Travel Query

Query the table as it existed at a specific snapshot.

In [ ]:
# Get the first snapshot ID
snapshots = spark.sql(f"SELECT snapshot_id FROM {catalog_name}.{schema_name}.{table_name}.snapshots ORDER BY committed_at").collect()

if len(snapshots) > 0:
    first_snapshot_id = snapshots[0]['snapshot_id']
    
    print(f"Querying table at first snapshot (ID: {first_snapshot_id})...")
    
    # Query using snapshot ID
    spark.sql(f"""
        SELECT customer_id, first_name, last_name, city, total_purchases 
        FROM {full_table} 
        VERSION AS OF {first_snapshot_id}
        ORDER BY customer_id
    """).show(truncate=False)
    
    print(f"\nCurrent table state (all snapshots):")
    spark.sql(f"""
        SELECT customer_id, first_name, last_name, city, total_purchases 
        FROM {full_table} 
        ORDER BY customer_id
    """).show(truncate=False)
else:
    print("No snapshots available for time travel query")

## 15. View Metadata Files

Examine the metadata files to understand Iceberg's internal structure.

In [ ]:
print(f"Metadata files for {full_table}:")
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.{table_name}.metadata_log_entries ORDER BY timestamp").show(truncate=False)

print(f"\nCurrent metadata location:")
spark.sql(f"DESCRIBE EXTENDED {full_table}").filter("col_name = 'Location'").show(truncate=False)

## 16. Summary

Display summary of all operations performed.

In [ ]:
print("="*80)
print("SUMMARY - Customer Table with Time Travel Demo")
print("="*80)

# Get final counts
total_customers = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table}").collect()[0]['cnt']
total_snapshots = spark.sql(f"SELECT COUNT(*) as cnt FROM {catalog_name}.{schema_name}.{table_name}.snapshots").collect()[0]['cnt']

print(f"\n✓ Catalog: {catalog_name}")
print(f"✓ Schema: {schema_name}")
print(f"✓ Table: {table_name}")
print(f"✓ Full reference: {full_table}")
print(f"\n✓ Total customers: {total_customers}")
print(f"✓ Total snapshots: {total_snapshots}")
print(f"\n✓ Retention policies configured:")
print(f"  - Maximum snapshot age: 7 days (604800000 ms)")
print(f"  - Minimum snapshots to keep: 5")
print(f"  - Manual expiration: Configured for snapshots older than 2024-11-01")
print(f"\n✓ Time travel capabilities: Enabled")
print(f"\nAll operations completed successfully!")

# List all tables in the schema
print(f"\nTables in {catalog_name}.{schema_name}:")
spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").show(truncate=False)

## Optional: Cleanup

Uncomment and run the following cell to drop the created table and schema.

In [ ]:
# # Uncomment to cleanup
# print("Cleaning up...")
# spark.sql(f"DROP TABLE IF EXISTS {full_table}")
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema_name} CASCADE")
# print("✓ Cleanup complete")